# **Análisis de precios.parquet**

El objetivo de este notebook es analizar el dataset precios.parquet para saber como tratar con los datos a la hora de crear los modelos.

# Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import spearmanr
from sklearn.preprocessing import QuantileTransformer

from src.utils.files import read_file
from src.utils.config import prices, load_env_file

## Carga de datos

In [ ]:
load_env_file()
use_minio = True 
minio = {"minio_write": False, "minio_read": use_minio}

df = read_file(prices, minio)
df['release_year'] = df['release_year'].astype(int)

df_cleaned = df.drop(columns=["id", "name", "v_resnet", "v_clip", "v_convnext"], errors='ignore').copy()

print("Dataset cargado")

En este DataFrame todos los juegos son no gratuitos, puesto que para el problema de predicción de precio sólo se tienen en cuenta juegos que no sean gratis.

---
# Análisis del dataset

## Análisis de nulos

In [ ]:
df_nulos = pd.DataFrame({
    'Total Nulos': df_cleaned.isnull().sum(),
    '% Nulos': round(df_cleaned.isnull().mean() * 100, 2)
}).sort_values('Total Nulos', ascending=False)

display(df_nulos[df_nulos['Total Nulos'] > 0])

plt.figure(figsize=(14, 6), facecolor='white')
sns.heatmap(df_cleaned.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title("Mapa de Nulos", color='black', fontsize=14)
plt.xticks(color='black', rotation=45, ha='right')
plt.tight_layout()
plt.show()

Observamos que hay algunos juegos de los que no se sabe cuantos juegos sacó su desarrollador/publisher antes de él. De momento no los borraremos, pero será algo a tener en cuenta.

In [ ]:
display(df[df.isnull().any(axis=1)][["name", "num_juegos_previos_publishers", "num_juegos_previos_developers"]])

Tras observar los juegos con nulos, nos damos cuenta de que son juegos sin un developer/publisher indicados en su página de Steam. Por ello, una buena forma de tratar con esto sería asumir que es developer y el publisher son lo mismo y copiar sus datos. La única excepción es Nefarious, el cual se deberá eliminar por no tener información de sus creadores en Steam.

In [ ]:
df_cleaned = df_cleaned.dropna(subset=["num_juegos_previos_developers", "num_juegos_previos_publishers"], how="all")

cols_devs = [
    "num_juegos_previos_developers", 
    "es_primer_juego_developers", 
    "ema_precio_developers", 
    "max_historico_precio_developers"
]

cols_pubs = [
    "num_juegos_previos_publishers", 
    "es_primer_juego_publishers", 
    "ema_precio_publishers", 
    "max_historico_precio_publishers"
]

filtro_devs = df_cleaned["num_juegos_previos_developers"].isna()
filtro_pubs = df_cleaned["num_juegos_previos_publishers"].isna()

df_cleaned.loc[filtro_devs, cols_devs] = df_cleaned.loc[filtro_devs, cols_pubs].values
df_cleaned.loc[filtro_pubs, cols_pubs] = df_cleaned.loc[filtro_pubs, cols_devs].values

In [ ]:
df_nulos = pd.DataFrame({
    'Total Nulos': df_cleaned.isnull().sum(),
    '% Nulos': round(df_cleaned.isnull().mean() * 100, 2)
}).sort_values('Total Nulos', ascending=False)

display(df_nulos[df_nulos['Total Nulos'] > 0])

## Separación de variables por cardinalidad

In [ ]:
continuas = [col for col in df_cleaned.columns if df_cleaned[col].nunique() > 2 and col not in ['price_overview', 'price_range']]
binarias = [col for col in df_cleaned.columns if df_cleaned[col].nunique() == 2]
constantes = [col for col in df_cleaned.columns if df_cleaned[col].nunique() <= 1]

print(f"Variables Continuas: {len(continuas)}")
print(f"Variables Binarias: {len(binarias)}")
print(f"Variables Constantes: {len(constantes)}")

## Análisis de variables de alto sesgo no informativo

In [ ]:
resultados = []
for col in binarias:
    conteos = df_cleaned[col].value_counts()
    if not conteos.empty:
        resultados.append({
            'Columna': col,
            'Valor_menos_frecuente': conteos.idxmin(),
            'Apariciones': conteos.min()
        })

df_resultado = pd.DataFrame(resultados).sort_values('Apariciones')
print(df_resultado[df_resultado['Apariciones'] < 100])
print()

df_filtro = df_resultado[df_resultado['Apariciones'] < 100]

for col, val in zip(df_filtro['Columna'], df_filtro['Valor_menos_frecuente']):
    print("-" * 40)
    print(f"JUEGOS CON {col} = {val}")
    print("-" * 40)
    print(df[df[col] == val][["name", "price_range", "price_overview"]])
    print()

Será necesario eliminar las variables "Free To Play" y "Family Sharing". La primera porque no hay juegos gratis en el dataset, por lo que se trata de un error en los datos; y la segunda porque la correlación entre Family Sharing y precios no es real, se trata de una anomalía estadística al momento de sacar el sample de 20k juegos, y muy seguramente no sea una relación que sigan el resto de juegos, por lo que solo generará ruido en nuestros modelos.

## Limpieza de datos

In [ ]:
erroneas = ["Family Sharing", "Free To Play"]
df_cleaned.drop(columns=erroneas, inplace=True, errors='ignore')

## Análisis de la variable respuesta

### Pie Chart de rango de precios

In [ ]:
fig = go.Figure()

orden = [
    "[0.01,4.99]",
    "[5.00,9.99]",
    "[10.00,14.99]",
    "[15.00,19.99]",
    "[20.00,29.99]",
    "[30.00,39.99]",
    ">40"
]

values_df = df["price_range"].value_counts()

values_df = values_df.reindex(orden)

fig.add_trace(go.Pie(values= values_df.values, labels= values_df.index, sort = False))

fig.update_layout(
                  paper_bgcolor = "#131416",
                  width = 800,
                  title = {
                      "text": "Proporción de los rangos de precio",
                      "x": 0.5,
                      "y":0.9,
                      "font":dict(family="Verdana", size = 20, weight="bold")
                  },
                  font_color = "white")

fig.show()

Casi el 50% de los juegos no gratuitos tienen un precio inferior a 5€.

### Pie Chart de la distribución de los géneros de cada rango de precios.

In [ ]:
genre_cols = ['Action','Adventure', 'Casual', 'Early Access', 'Indie', 'RPG', 'Simulation',
       'Strategy']

In [ ]:
df_long = (
    df[["price_range"] + genre_cols]
    .melt(
        id_vars="price_range",
        value_vars=genre_cols,
        var_name="genre",
        value_name="present"
    )
)

df_long = df_long[df_long["present"] == 1].copy()


df_long["genres"] = df_long["genre"].str.replace("genres_", "", regex=False)


price_genre_counts = (
    df_long.groupby(["price_range", "genres"])
    .size()
    .unstack(fill_value=0)
)

fig = make_subplots(
    rows=2, cols=4,
    specs=[
        [{"type": "domain"}, {"type": "domain"}, {"type": "domain"}, {"type": "domain"}],
        [{"type": "domain"}, {"type": "domain"}, {"type": "domain"}, {"type": "domain"}]
    ],
    subplot_titles=[
        "[0.01,4.99]", "[5.00,9.99]", "[10.00,14.99]",
        "[15.00,19.99]", "[20.00,29.99]", "[30.00,39.99]", ">40"
    ]
)

orden = {
    "[0.01,4.99]": [1, 1],
    "[5.00,9.99]": [1, 2],
    "[10.00,14.99]": [1, 3],
    "[15.00,19.99]": [1, 4],
    "[20.00,29.99]": [2, 1],
    "[30.00,39.99]": [2, 2],
    ">40": [2, 3]
}

N = 5 

for rango_precio, pos in orden.items():
    if rango_precio not in price_genre_counts.index:
        continue

    genre_counts = price_genre_counts.loc[rango_precio].sort_values(ascending=False).head(N)

    if not genre_counts.empty:
        fig.add_trace(
            go.Pie(
                values=genre_counts.values,
                labels=genre_counts.index,
                sort=False,
                name=rango_precio,
                textposition="inside"
            ),
            row=pos[0], col=pos[1]
        )

fig.update_layout(
    paper_bgcolor="#131416",
    font_color="white",
    title={
        "text": "Distribución de los géneros por cada rango de precio",
        "x": 0.5,
        "y": 0.93
    },
    legend_title="Géneros",
    legend=dict(
        x=0.87,
        y=-0.1,
        font=dict(size=10)
    )
)

fig.show()

- Se puede apreciar que los porcentajes de los géneros más populares de cada rango varían significativamente. 

- El ejemplo más claro sería el género "Indie". Como se mencionó anteriormente, los juegos de este género abundan en los rangos de precio inferiores y
a medida que aumenta el precio su porcentaje en cada rango disminuye.

- Otro ejemplo bastante representativo sería el del género "Casual". Los juegos de este género suelen ser juegos sencillos y no muy largos, haciéndolos accesibles a la mayor cantidad de público posible. Como resultado, muchos de estos juegos se encuentran relacionados con los juegos "Indie", y están más alejados de juegos tipo "Estrategia" o "Acción" que suelen abundar en las grandes producciones (juegos "AAA"). Es por ello que el porcentaje de juegos "Casual" disminuye a medida que aumenta el precio.

## Análisis de multicolinealidad

### Calculamos el FIV

In [ ]:
def calcular_fiv(df_input):
    X = df_input.select_dtypes(include=[np.number]).drop(columns=['price_overview', 'price_range'], errors='ignore')
    X = X.fillna(X.median())
    
    X_const = sm.add_constant(X) # Añadimos el intercepto
    fiv_data = pd.DataFrame()
    fiv_data["Variable"] = X_const.columns
    fiv_data["FIV"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
    return fiv_data[fiv_data['Variable'] != 'const'].sort_values('FIV', ascending=False)

vif_df = calcular_fiv(df_cleaned)
display(vif_df[vif_df['FIV'] > 5])

Observamos una multicolinealidad significativa en:
- "max_historico_precio_publishers"
- "max_historico_precio_developers"
- "ema_precio_publishers"
- "ema_precio_developers"

y una multicolinealidad leve en:
- "Multi-player"
- "es_primer_juego_publishers"
- "es_primer_juego_developers"

### Vamos a ver un clustermap, calculado con el coeficiente de correlación lineal de Pearson

In [ ]:
plt.figure(figsize=(10, 8))
sns.clustermap(df_cleaned[continuas].corr(method='spearman'), annot=True, cmap="coolwarm", facecolor='white'
               ,vmin=-1, vmax=1)
plt.show()

In [ ]:
plt.figure(figsize=(30, 24))
sns.clustermap(df_cleaned.drop(columns=['price_range']).corr(method='spearman'), annot=False, cmap="coolwarm", facecolor='white'
               ,vmin=-1, vmax=1)
plt.show()

Observamos que las variables que están relacionadas con el historial de los creadores están bastante relacionadas entre sí.

## Detección de outliers
Usaremos escala logarítmica para mejorar la visibilidad

In [ ]:
fig = go.Figure()

for col in continuas:
    fig.add_trace(go.Box(
        x=df_cleaned[col],
        name=col,
        orientation='h'
    ))

fig.update_layout(
    title="Outliers en variables continuas (Escala Logarítmica)",
    xaxis=dict(type='log'),
    plot_bgcolor='white',
    showlegend=False,
    height=100 + len(continuas) * 55
)

fig.show()

Observamos que las colas son tan largas que en muchas variables hay muchísimos outliers

## Análisis de skew y kurtosis

In [ ]:
extreme_skews = []
extreme_kurtosis = []

df_numeric = df_cleaned[continuas]

print(f"{'COLUMNA':<32} | {'SKEW':>10} | {'CURTOSIS':>10}")
print("-" * 60)

for col in df_numeric.columns:
    skew = df_numeric[col].skew()
    kurtosis = df_numeric[col].kurtosis()

    print(f"{col:<32} | {skew:>10.3f} | {kurtosis:>10.3f}")

    if skew < -1 or skew > 1:
        extreme_skews.append(col)
    if kurtosis > 2: 
        extreme_kurtosis.append(col)


print(f"\nSkew extremo:      {', '.join(extreme_skews) if extreme_skews else 'Ninguno'}")
print(f"Curtosis extrema:  {', '.join(extreme_kurtosis) if extreme_kurtosis else 'Ninguno'}")

## Análisis de normalidad

In [ ]:
normales = []
no_normales = []

for col in continuas:
    data = df_cleaned[col].dropna()
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Distribución
    sns.histplot(data, kde=True, ax=axes[0])
    axes[0].set_title(f"Distribución: {col}")
    
    # Q-Q Plot
    stats.probplot(data, dist="norm", plot=axes[1])
    axes[1].set_title(f"Q-Q Plot: {col}")
    
    # Test estadístico
    _, p_val = stats.normaltest(data)
    plt.suptitle(f"Variable: {col}")
    plt.show()
    
    if p_val >= 0.05:
        resultado = "No se rechaza H0 (distribución normal)"
        normales.append(col)
    else:
        resultado = "Se rechaza H0 (no es normal)"
        no_normales.append(col)
    
    print(f"Variable: {col}")
    print(f"p-valor: {p_val:.4f}")
    print(f"{resultado}")

print("\nRESULTADOS:")
print(f"    Variables normales: {normales}")
print(f"    Variables no normales: {no_normales}")

Podemos observar que ninguna variable sigue una distribución normal, y en muchos casos las colas son enormes

## Análisis de las distribuciones

Vamos a observar las distribuciones de las variables numéricas de forma mas limpia y compacta

In [ ]:
columns = df_cleaned.columns.tolist()
n_cols = 4
n_rows = math.ceil(len(continuas) / n_cols)

fig = make_subplots(
    rows=n_rows, 
    cols=n_cols, 
    subplot_titles=continuas,
    vertical_spacing=0.05,
    horizontal_spacing=0.05
)

for i, col in enumerate(continuas):
    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1
    
    fig.add_trace(
        go.Histogram(
            x=df_cleaned[col], 
            name=col, 
            marker=dict(color="#69C0EC", line=dict(color='black', width=1))
        ),
        row=row, 
        col=col_pos
    )

fig.update_layout(
    title_text="Distribución de las variables de entrada (X)",
    title_x=0.5,
    title_font_size=20,
    height=300 * n_rows,
    showlegend=False,
    plot_bgcolor='white'
)

fig.show()

Confirmamos lo que ya habíamos visto en el análisis de la normalidad y del skew y curtosis, ninguna variable es normal y salvo "description_len", "release_year" y "brillo" las colas son enormes.

## Análisis bivariante
Estudiaremos la relación con la variable respuesta

In [ ]:
for col in continuas:
    plt.figure(figsize=(10, 5), facecolor='white')
    sns.boxplot(data=df_cleaned, x='price_range', y=col, order=orden)
    
    # Aplicar log si hay mucha asimetría para ver mejor la caja
    if df_cleaned[col].skew() > 2:
        plt.yscale('log')
        
    plt.title(f"Impacto de {col} en el Rango de Precio")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
correlaciones = df_cleaned.drop(columns=["price_range", "price_overview"]).corrwith(
    df_cleaned['price_overview'], method='spearman').sort_values() # Usamos spearman porque no asume normalidad

fig1 = go.Figure(go.Bar(
    x=correlaciones.values,
    y=correlaciones.index,
    orientation='h',
    marker_color=['#FFA43B' if x < 0 else "#3BA4FF" for x in correlaciones.values]
))
fig1.add_vline(x=0, line_width=1, line_color='black')
fig1.update_layout(title="Correlación con price_overview", template='plotly_white', height=700)
fig1.show()

resultados = {}
for col in continuas:
    r, p = spearmanr(df_cleaned[col], df_cleaned['price_overview'])
    resultados[col] = {'r (coeficiente de Spearman)': r, 'p-valor': p}

pd.DataFrame(resultados).T.sort_values('r (coeficiente de Spearman)', ascending=False)

Observamos algunas correlaciones cercanas a 0 en varias variables, y la mayoría de las correlaciones son entre moderadas e inexistentes

- - -

### Conclusión general: 

- La distribución de la variable respuesta es una cola larga, con una gran concentración de juegos en los precios bajos (en gran parte debido a la cantidad de juegos "Indie"), y una minoría superando la barrera de los juegos "AAA".

- Es importante notar que dentro de cada rango de precio las proporciones de géneros varían significativamente y de manera coherente.

- Hay géneros que se mantienen bastante constantes a lo largo de los distintos rangos de precio (ej: "Aventura"), por otro lado existen géneros que se localizan principalmente en rangos específicos (ej: "Indie", "Casual").

- Existen muy pocos nulos, pero se deben tratar a la hora de hacer los modelos.

- Hay correlaciones bastante fuertes entre las variables que contienen información del historial de precios de los developers/publishers y entre todas las variables que contienen información del developers con su análogo del publisher. Además, la variable multijugador introduce multicolinealidad leve.

- Hay muchas variables con unas skew y curtosis altisimas, se deberá tener en cuenta para transformar las variables.

- Ninguna variable sigue una distribución normal, las únicas que están cerca de asemejarse son "description_len", "release_year" y "brillo".

- La mayoría de las variables apenas tienen correlación con la variable respuesta